In [17]:
import os
import pandas as pd
import re

# Extract class from text
def extract_class(text):
    text = str(text).lower()
    if 'k-7' in text:
        return 'K-7'
    elif 'k-9' in text or 'k9' in text:
        return 'K-9'
    elif 'm.s' in text or 'ms' in text:
        return 'M.S'
    elif 'hdpe' in text:
        return 'HDPE'
    elif 'upvc' in text:
        return 'UPVC'
    elif 'pvc' in text:
        return 'PVC'
    elif 'ci' in text:
        return 'CI'
    elif 'di' in text:
        return 'DI'
    else:
        return None

# Extract diameter
def extract_dia(text):
    match = re.search(r'(\d+)\s*mm', str(text).lower())
    return int(match.group(1)) if match else None

# Column detector
def find_column(columns, keywords):
    for col in columns:
        if any(k in str(col).lower() for k in keywords):
            return col
    return None

# Main processor
def process_boq_file(file_path, output_dir):
    filename = os.path.basename(file_path)
    print(f"\n🔍 Processing: {filename}")

    try:
        df_raw = pd.read_excel(file_path, header=None, engine="xlrd")
    except Exception as e:
        print(f"❌ Failed to read {filename}: {e}")
        return

    # Guess header row
    header_row_index = None
    for i in range(min(15, len(df_raw))):
        row = df_raw.iloc[i].astype(str).str.lower()
        if any("desc" in cell or "item" in cell for cell in row) and any("unit" in cell for cell in row):
            header_row_index = i
            break

    if header_row_index is None:
        print(f"⚠️ No header row found in {filename}")
        return

    df = pd.read_excel(file_path, header=header_row_index, engine="xlrd")

    # Detect relevant columns
    desc_col = find_column(df.columns, ['desc', 'item'])
    unit_col = find_column(df.columns, ['unit'])
    name_of_work_col = find_column(df.columns, ['name of work', 'work'])

    if not desc_col or not unit_col:
        print(f"⚠️ Missing required columns in {filename}")
        return

    # Normalize units
    df[unit_col] = df[unit_col].astype(str).str.strip().str.lower().str.replace('.', '', regex=False)

    # Switch-case like matching
    def is_pipe_row(row):
        desc = str(row[desc_col]).lower()
        unit = str(row[unit_col]).lower()
        name_of_work = str(row[name_of_work_col]).lower() if name_of_work_col else ""

        if "pipe" in desc:
            return True
        elif any(cls.lower() in desc for cls in ["di", "ci", "m.s", "k-7", "k-9", "hdpe", "pvc", "upvc"]):
            return True
        elif "pipe" in name_of_work or any(cls in name_of_work for cls in ["di", "ci", "m.s", "k-7", "k-9", "hdpe", "pvc", "upvc"]):
            return True
        else:
            return False

    filtered_df = df[df.apply(is_pipe_row, axis=1)].copy()

    if filtered_df.empty:
        print(f"ℹ️ No matching rows in {filename}")
        print("🔍 Sample Descriptions:", df[desc_col].dropna().astype(str).str.strip().unique()[:5].tolist())
        print("🔍 Sample Units:", df[unit_col].dropna().astype(str).str.strip().unique()[:5].tolist())
        return

    # Add cleaned columns
    filtered_df["Item Description"] = filtered_df[desc_col].astype(str)
    filtered_df["class"] = filtered_df["Item Description"].apply(extract_class)
    filtered_df["DIA"] = filtered_df["Item Description"].apply(extract_dia)

    # Estimate Rate
    rate_col = find_column(df.columns, ['rate', 'amount', 'estimate'])
    filtered_df["estimate rate"] = pd.to_numeric(filtered_df[rate_col], errors='coerce') if rate_col else 0
    filtered_df["Units"] = filtered_df[unit_col].astype(str).str.strip()

    # Finalize
    final_df = filtered_df[["Item Description", "class", "DIA", "estimate rate", "Units"]].copy()
    final_df.insert(0, "SL No", range(1, len(final_df) + 1))

    # Save
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, os.path.splitext(filename)[0] + "_filter.xlsx")
    final_df.to_excel(output_path, index=False, engine="openpyxl")
    print(f"✅ Saved: {output_path}")

# Batch process
def process_all_xls_files(folder="."):
    output_dir = os.path.join(folder, "filtered_results")
    os.makedirs(output_dir, exist_ok=True)
    for file in os.listdir(folder):
        if file.lower().endswith(".xls"):
            process_boq_file(os.path.join(folder, file), output_dir)

# Run
process_all_xls_files(".")



🔍 Processing: BOQ_115246.xls
✅ Saved: .\filtered_results\BOQ_115246_filter.xlsx

🔍 Processing: BOQ_1833226.xls
✅ Saved: .\filtered_results\BOQ_1833226_filter.xlsx

🔍 Processing: BOQ_1844315.xls
✅ Saved: .\filtered_results\BOQ_1844315_filter.xlsx

🔍 Processing: BOQ_1845092.xls
✅ Saved: .\filtered_results\BOQ_1845092_filter.xlsx

🔍 Processing: BOQ_1868265.xls
✅ Saved: .\filtered_results\BOQ_1868265_filter.xlsx

🔍 Processing: BOQ_1886848.xls
✅ Saved: .\filtered_results\BOQ_1886848_filter.xlsx
